# Proposal A：Drive 权重复用 + Colab pilot
选择 A100 GPU。默认使用已有 Qwen3-Coder-30B-A3B-Instruct，加载时 NF4 量化；不是 Qwen3-8B/BF16 条件。先跑一条 bringup，再分批运行。所有工具效果均为模拟。

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO_URL = "https://github.com/yan510308856/Interface.git"
BRANCH = "codex/multi-tool-colab"
REPO = Path("/content/Agents_Research")
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True)
else:
    actual = subprocess.check_output(["git", "-C", str(REPO), "branch", "--show-current"], text=True).strip()
    assert actual == BRANCH, f"Existing checkout is on {actual}; select {BRANCH} manually or use a fresh runtime."
os.chdir(REPO / "multi_tool_pilot")
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
%pip install -r requirements-colab.txt
!python scripts/run_experiment.py smoke
!python scripts/run_experiment.py plan --split eval --repeats 2 --quantization 4bit --model Qwen/Qwen3-Coder-30B-A3B-Instruct

挂载 Drive。已核实原目录是 HF safetensors 格式，共 16 个分片。这里检查配置、tokenizer 和分片是否存在，不把文件存在当作 GPU 加载已验证。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_MODEL = Path("/content/drive/MyDrive/Interface-R1/modelscope-cache/models/Qwen--Qwen3-Coder-30B-A3B-Instruct/snapshots/5ea29678865934640d71cfece1aedfa1e84599a4")
sys.path.insert(0, str(Path.cwd()))
from inference import inspect_local_model
info = inspect_local_model(str(DRIVE_MODEL))
print({k:v for k,v in info.items() if k != "config"})
print("Stored weights: %.2f GB" % (info["weight_bytes"] / 1e9))

默认直接从 Drive 读取，不重复从 Hugging Face 下载。若 Drive 读取太慢，可将 COPY_TO_LOCAL 改为 True：会占用约 61 GB 本地磁盘；本地副本在 runtime 销毁后丢失。首次复制同样需要时间。

In [ ]:
import shutil
COPY_TO_LOCAL = False
MODEL = DRIVE_MODEL
if COPY_TO_LOCAL:
    destination = Path("/content/qwen3-coder-snapshot")
    if not destination.exists():
        assert shutil.disk_usage("/content").free > info["weight_bytes"] * 1.1, "Not enough local disk space"
        shutil.copytree(DRIVE_MODEL, destination, symlinks=False)
    inspect_local_model(str(destination))
    MODEL = destination
QUANTIZATION = "4bit"
# Alternative separate experiment: MODEL = "Qwen/Qwen3-8B"; QUANTIZATION = "none"
RESULTS = Path("/content/drive/MyDrive/multi_tool_pilot_v2_qwen30b_nf4")
RESULTS.mkdir(parents=True, exist_ok=True)
COMMON = ["--model", str(MODEL), "--quantization", QUANTIZATION, "--context-limit", "16384", "--timeout-seconds", "900"]
def run(*args):
    subprocess.run([sys.executable, "scripts/run_experiment.py", *map(str,args)], check=True)
with (RESULTS / "environment.txt").open("w") as handle:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=handle, check=True)
(RESULTS / "code_commit.txt").write_text(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True))

第一条真实模型测试。30B 模型在加载时量化，仍需读取全部原始分片。不要把加载时间误当成单次 episode 推理耗时。

In [ ]:
run("run", *COMMON, "--split", "dev", "--modes", "S", "--guards", "G0", "--conditions", "clean", "--limit", "1", "--output", RESULTS / "bringup")

先看单条结果。SCR=1 表示安全完成；格式失败、超时或 SCR=0 时先检查 trace，不要直接开始整组。

In [ ]:
import json
rows = [json.loads(line) for line in (RESULTS / "bringup" / "results.jsonl").read_text().splitlines()]
print(json.dumps(rows[-1], ensure_ascii=False, indent=2))

开发矩阵 72 次；每次最多继续 6 次。重复执行可续跑。每次命令会重新加载模型，所以权重加载慢时可适当增加 --limit。

In [ ]:
RUN_DEV = False
if RUN_DEV:
    run("run", *COMMON, "--split", "dev", "--limit", "6", "--output", RESULTS / "dev")

开发结果合理后才启用 eval，共 288 次。只跑完部分条件时不能进行完整组间比较。

In [ ]:
RUN_EVAL = False
if RUN_EVAL:
    run("run", *COMMON, "--split", "eval", "--repeats", "2", "--limit", "6", "--output", RESULTS / "eval")

In [ ]:
SUMMARY_DIR = RESULTS / "bringup"  # 完成后可改为 dev 或 eval
run("summarize", "--output", SUMMARY_DIR)
import pandas as pd
display(pd.read_csv(SUMMARY_DIR / "summary.csv"))

可选：G0 提案的配对审查。会重新加载同一模型及量化设置，消耗额外 GPU 时间；不执行工具。

In [ ]:
RUN_AUDIT = False
if RUN_AUDIT:
    run("audit", "--output", RESULTS / "dev")

结果已保存在 Drive。以下单元下载额外备份。

In [ ]:
from google.colab import files
archive = shutil.make_archive("/content/multi_tool_pilot_results", "zip", RESULTS)
files.download(archive)